# Accepted Loan Final Model Comparison

Compare the selected Logistic Regression, Random Forest, and HistGradientBoosting models using their per-model notebook outputs.

## 1. Setup

In [1]:

from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "Cleaning").exists() and (candidate / "Modeling").exists():
            return candidate
    raise FileNotFoundError("Could not find CreditRiskRAG project root")

PROJECT_ROOT = find_project_root()
MODELING_OUTPUT_ROOT = PROJECT_ROOT / "Modeling" / "modeling_outputs"
FINAL_OUTPUT_ROOT = MODELING_OUTPUT_ROOT / "final_comparison"
TABLE_DIR = FINAL_OUTPUT_ROOT / "tables"
PLOT_DIR = FINAL_OUTPUT_ROOT / "plots"
for directory in [TABLE_DIR, PLOT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

MODEL_FAMILIES = ["logistic_regression", "random_forest", "hist_gradient_boosting"]
MODEL_LABELS = {
    "logistic_regression": "Logistic Regression",
    "random_forest": "Random Forest",
    "hist_gradient_boosting": "HistGradientBoosting",
}

def save_table(df: pd.DataFrame, name: str) -> Path:
    path = TABLE_DIR / f"final_model_{name}.csv"
    df.to_csv(path, index=False)
    print("Saved:", path)
    return path

def save_plot(fig, name: str) -> Path:
    path = PLOT_DIR / f"final_model_{name}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("Saved:", path)
    return path

print("Project root:", PROJECT_ROOT)
print("Final comparison outputs:", FINAL_OUTPUT_ROOT)


Project root: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG
Final comparison outputs: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison


## 2. Load Per-Model Outputs

In [2]:

def read_model_table(model_family: str, table_name: str) -> pd.DataFrame:
    path = MODELING_OUTPUT_ROOT / model_family / "tables" / f"{model_family}_{table_name}.csv"
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}. Run the {model_family} notebook first.")
    return pd.read_csv(path)

candidate_tables = []
selected_metric_tables = []
review_volume_tables = []
selected_candidate_tables = []
for model_family in MODEL_FAMILIES:
    candidate_tables.append(read_model_table(model_family, "candidate_results"))
    selected_metric_tables.append(read_model_table(model_family, "selected_model_metrics"))
    review_volume_tables.append(read_model_table(model_family, "review_volume_precision"))
    selected_candidate_tables.append(read_model_table(model_family, "selected_candidate"))

all_candidates = pd.concat(candidate_tables, ignore_index=True)
selected_metrics = pd.concat(selected_metric_tables, ignore_index=True)
review_volume_precision = pd.concat(review_volume_tables, ignore_index=True)
selected_candidates = pd.concat(selected_candidate_tables, ignore_index=True)

for frame in [all_candidates, selected_metrics, review_volume_precision, selected_candidates]:
    frame["model_label"] = frame["model_family"].map(MODEL_LABELS)

save_table(all_candidates, "candidate_summary")
save_table(selected_metrics, "selected_metrics")
save_table(review_volume_precision, "review_volume_precision")
save_table(selected_candidates, "selected_candidates")
display(selected_candidates)
display(selected_metrics)


Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/tables/final_model_candidate_summary.csv
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/tables/final_model_selected_metrics.csv
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/tables/final_model_review_volume_precision.csv
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/tables/final_model_selected_candidates.csv


,model_family,candidate,params,fit_rows,fit_bad_rate,fit_seconds,roc_auc,pr_auc,best_f1_threshold,best_f1,best_f1_precision,best_f1_recall,target_precision_threshold,target_precision_f1,target_precision,target_precision_recall,model_label
0,logistic_regression,logistic_regression_07,"{""C"": 0.5, ""class_weight"": ""balanced"", ""l1_rat...",962641,0.1883,39.346,0.695407,0.412336,0.471801,0.470644,0.355089,0.697691,0.563511,0.447766,0.400000,0.508488,Logistic Regression
1,random_forest,random_forest_03,"{""class_weight"": ""balanced_subsample"", ""max_de...",200000,0.1883,55.434,0.694572,0.415081,0.455266,0.468385,0.359529,0.671783,0.523016,0.447541,0.400003,0.507902,Random Forest
2,hist_gradient_boosting,hist_gradient_boosting_04,"{""l2_regularization"": 0.0, ""learning_rate"": 0....",300000,0.1883,27.427,0.700829,0.423500,0.472937,0.474259,0.359479,0.696715,0.550043,0.458965,0.400000,0.538320,HistGradientBoosting


,model_family,model,split,operating_point,rows,bad_rate,threshold,roc_auc,pr_auc,brier_score,precision,recall,f1,tn,fp,fn,tp,model_label
0,logistic_regression,logistic_regression_07,train,best_validation_f1,962641,0.188300,0.471801,0.719952,0.372912,0.214029,0.294505,0.713552,0.416930,471534,309842,51923,129342,Logistic Regression
1,logistic_regression,logistic_regression_07,train,target_validation_precision,962641,0.188300,0.563511,0.719952,0.372912,0.214029,0.345530,0.537793,0.420738,596733,184643,83782,97483,Logistic Regression
2,logistic_regression,logistic_regression_07,validation,best_validation_f1,186920,0.246763,0.471801,0.695407,0.412336,0.224051,0.355082,0.697669,0.470633,82348,58447,13945,32180,Logistic Regression
3,logistic_regression,logistic_regression_07,validation,target_validation_precision,186920,0.246763,0.563511,0.695407,0.412336,0.224051,0.400007,0.508488,0.447771,105615,35180,22671,23454,Logistic Regression
4,logistic_regression,logistic_regression_07,test,best_validation_f1,195749,0.210315,0.471801,0.700061,0.361509,0.225017,0.310541,0.709320,0.431966,89746,64834,11967,29202,Logistic Regression
5,logistic_regression,logistic_regression_07,test,target_validation_precision,195749,0.210315,0.563511,0.700061,0.361509,0.225017,0.348254,0.538026,0.422823,113127,41453,19019,22150,Logistic Regression
6,random_forest,random_forest_03,train,best_validation_f1,962641,0.188300,0.455266,0.738405,0.398883,0.194754,0.309538,0.717259,0.432450,491364,290012,51251,130014,Random Forest
7,random_forest,random_forest_03,train,target_validation_precision,962641,0.188300,0.523016,0.738405,0.398883,0.194754,0.355784,0.574358,0.439390,592863,188513,77154,104111,Random Forest
8,random_forest,random_forest_03,validation,best_validation_f1,186920,0.246763,0.455266,0.694572,0.415081,0.207616,0.359521,0.671762,0.468373,85596,55199,15140,30985,Random Forest
9,random_forest,random_forest_03,validation,target_validation_precision,186920,0.246763,0.523016,0.694572,0.415081,0.207616,0.400010,0.507902,0.447546,105656,35139,22698,23427,Random Forest


## 3. Validation And Test Comparison

In [3]:

comparison = selected_metrics[
    selected_metrics["split"].isin(["validation", "test"])
].copy()
comparison = comparison.sort_values(["operating_point", "split", "f1", "precision", "pr_auc"], ascending=[True, True, False, False, False])
save_table(comparison, "validation_test_comparison")
display(comparison)

for metric in ["f1", "precision", "recall", "pr_auc", "roc_auc", "brier_score"]:
    plot_df = comparison[comparison["operating_point"] == "best_validation_f1"]
    fig, ax = plt.subplots(figsize=(9, 4.5))
    width = 0.35
    labels = [MODEL_LABELS[m] for m in MODEL_FAMILIES]
    x = np.arange(len(labels))
    val = plot_df[plot_df["split"] == "validation"].set_index("model_family").loc[MODEL_FAMILIES, metric]
    test = plot_df[plot_df["split"] == "test"].set_index("model_family").loc[MODEL_FAMILIES, metric]
    ax.bar(x - width/2, val, width, label="validation")
    ax.bar(x + width/2, test, width, label="test")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=20, ha="right")
    ax.set_title(f"Final model comparison: {metric}")
    ax.grid(axis="y", alpha=0.25)
    ax.legend()
    save_plot(fig, f"{metric}_comparison")


Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/tables/final_model_validation_test_comparison.csv


,model_family,model,split,operating_point,rows,bad_rate,threshold,roc_auc,pr_auc,brier_score,precision,recall,f1,tn,fp,fn,tp,model_label
16,hist_gradient_boosting,hist_gradient_boosting_04,test,best_validation_f1,195749,0.210315,0.472937,0.708493,0.377185,0.212358,0.319695,0.693143,0.437571,93856,60724,12633,28536,HistGradientBoosting
10,random_forest,random_forest_03,test,best_validation_f1,195749,0.210315,0.455266,0.703753,0.371073,0.200271,0.321114,0.673832,0.434952,95931,58649,13428,27741,Random Forest
4,logistic_regression,logistic_regression_07,test,best_validation_f1,195749,0.210315,0.471801,0.700061,0.361509,0.225017,0.310541,0.709320,0.431966,89746,64834,11967,29202,Logistic Regression
14,hist_gradient_boosting,hist_gradient_boosting_04,validation,best_validation_f1,186920,0.246763,0.472937,0.700829,0.423500,0.217717,0.359479,0.696715,0.474259,83535,57260,13989,32136,HistGradientBoosting
2,logistic_regression,logistic_regression_07,validation,best_validation_f1,186920,0.246763,0.471801,0.695407,0.412336,0.224051,0.355082,0.697669,0.470633,82348,58447,13945,32180,Logistic Regression
8,random_forest,random_forest_03,validation,best_validation_f1,186920,0.246763,0.455266,0.694572,0.415081,0.207616,0.359521,0.671762,0.468373,85596,55199,15140,30985,Random Forest
17,hist_gradient_boosting,hist_gradient_boosting_04,test,target_validation_precision,195749,0.210315,0.550043,0.708493,0.377185,0.212358,0.357576,0.543856,0.431469,114354,40226,18779,22390,HistGradientBoosting
11,random_forest,random_forest_03,test,target_validation_precision,195749,0.210315,0.523016,0.703753,0.371073,0.200271,0.359378,0.520076,0.425045,116413,38167,19758,21411,Random Forest
5,logistic_regression,logistic_regression_07,test,target_validation_precision,195749,0.210315,0.563511,0.700061,0.361509,0.225017,0.348254,0.538026,0.422823,113127,41453,19019,22150,Logistic Regression
15,hist_gradient_boosting,hist_gradient_boosting_04,validation,target_validation_precision,186920,0.246763,0.550043,0.700829,0.423500,0.217717,0.400006,0.538320,0.458969,103551,37244,21295,24830,HistGradientBoosting


Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/plots/final_model_f1_comparison.png
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/plots/final_model_precision_comparison.png
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/plots/final_model_recall_comparison.png


Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/plots/final_model_pr_auc_comparison.png
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/plots/final_model_roc_auc_comparison.png


Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/plots/final_model_brier_score_comparison.png


## 4. Fixed Review Volume Comparison

In [4]:

review_compare = review_volume_precision[review_volume_precision["split"].isin(["validation", "test"])].copy()
save_table(review_compare, "review_volume_comparison")
display(review_compare)

for split in ["validation", "test"]:
    plot_df = review_compare[review_compare["split"] == split]
    fig, ax = plt.subplots(figsize=(8.5, 4.5))
    for model_family, group in plot_df.groupby("model_family"):
        ax.plot(group["review_pct"], group["precision"], marker="o", label=MODEL_LABELS[model_family])
    ax.set_title(f"Precision at fixed review volumes: {split}")
    ax.set_xlabel("Reviewed applications (%)")
    ax.set_ylabel("Precision")
    ax.grid(alpha=0.25)
    ax.legend()
    save_plot(fig, f"precision_by_review_volume_{split}")


Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/tables/final_model_review_volume_comparison.csv


,model_family,model,split,review_pct,review_count,captured_bad,precision,recall,base_bad_rate,lift_over_base_bad_rate,model_label
0,logistic_regression,logistic_regression_07,validation,1.0,1870,1132,0.605348,0.024542,0.246763,2.453151,Logistic Regression
1,logistic_regression,logistic_regression_07,validation,2.0,3739,2174,0.581439,0.047133,0.246763,2.356261,Logistic Regression
2,logistic_regression,logistic_regression_07,validation,5.0,9346,5073,0.542799,0.109984,0.246763,2.199675,Logistic Regression
3,logistic_regression,logistic_regression_07,validation,10.0,18692,9278,0.496362,0.201149,0.246763,2.011491,Logistic Regression
4,logistic_regression,logistic_regression_07,validation,15.0,28038,13006,0.463870,0.281973,0.246763,1.879819,Logistic Regression
5,logistic_regression,logistic_regression_07,validation,20.0,37384,16488,0.441044,0.357463,0.246763,1.787317,Logistic Regression
6,logistic_regression,logistic_regression_07,validation,25.0,46730,19675,0.421036,0.426558,0.246763,1.706233,Logistic Regression
7,logistic_regression,logistic_regression_07,validation,30.0,56076,22690,0.404629,0.491924,0.246763,1.639747,Logistic Regression
8,logistic_regression,logistic_regression_07,test,1.0,1958,1018,0.519918,0.024727,0.210315,2.472090,Logistic Regression
9,logistic_regression,logistic_regression_07,test,2.0,3915,2000,0.510856,0.048580,0.210315,2.429000,Logistic Regression


Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/plots/final_model_precision_by_review_volume_validation.png


Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/plots/final_model_precision_by_review_volume_test.png


## 5. Grade/Subgrade Ablation

Compare the current baseline feature set against the `baseline_no_grade_subgrade` feature set. This tests whether LendingClub grade/subgrade encoded fields are driving model performance.

In [5]:

NO_GRADE_METRICS_PATH = MODELING_OUTPUT_ROOT / "tables" / "no_grade_subgrade_model_metrics.csv"
NO_GRADE_EXPORT_SUMMARY_PATH = PROJECT_ROOT / "Modeling" / "Preprocessing" / "preprocessing_outputs" / "tables" / "preprocessing_no_grade_subgrade_export_summary.csv"

if not NO_GRADE_METRICS_PATH.exists():
    raise FileNotFoundError(
        f"Missing {NO_GRADE_METRICS_PATH}. Run the no-grade/subgrade modeling export first."
    )

no_grade_metrics = pd.read_csv(NO_GRADE_METRICS_PATH)
no_grade_metrics = no_grade_metrics[no_grade_metrics["split"].isin(["validation", "test"])].copy()
no_grade_metrics["model_family"] = (
    no_grade_metrics["model"]
    .str.replace("_no_grade_subgrade", "", regex=False)
    .str.replace("hist_gradient_boosting", "hist_gradient_boosting", regex=False)
)
no_grade_metrics["model_label"] = no_grade_metrics["model_family"].map(MODEL_LABELS)
no_grade_metrics["feature_set"] = "baseline_no_grade_subgrade"

with_grade_metrics = selected_metrics[
    (selected_metrics["split"].isin(["validation", "test"])) &
    (selected_metrics["operating_point"] == "best_validation_f1")
].copy()
with_grade_metrics["feature_set"] = "baseline_with_grade_subgrade"
with_grade_metrics = with_grade_metrics[[
    "model_family", "model_label", "model", "split", "feature_set",
    "roc_auc", "pr_auc", "brier_score", "precision", "recall", "f1", "threshold"
]]

no_grade_metrics = no_grade_metrics[[
    "model_family", "model_label", "model", "split", "feature_set",
    "roc_auc", "pr_auc", "brier_score", "precision", "recall", "f1", "threshold"
]]

ablation_metrics = pd.concat([with_grade_metrics, no_grade_metrics], ignore_index=True)
save_table(ablation_metrics, "grade_subgrade_ablation_metrics")
display(ablation_metrics.sort_values(["split", "model_label", "feature_set"]))

wide = ablation_metrics.pivot_table(
    index=["model_family", "model_label", "split"],
    columns="feature_set",
    values=["roc_auc", "pr_auc", "f1", "precision", "recall"],
    aggfunc="first",
)
wide.columns = [f"{metric}_{feature_set}" for metric, feature_set in wide.columns]
wide = wide.reset_index()

for metric in ["roc_auc", "pr_auc", "f1", "precision", "recall"]:
    wide[f"{metric}_delta"] = wide[f"{metric}_baseline_no_grade_subgrade"] - wide[f"{metric}_baseline_with_grade_subgrade"]

ablation_deltas = wide.sort_values(["split", "model_label"])
save_table(ablation_deltas, "grade_subgrade_ablation_deltas")
display(ablation_deltas)

compact_cols = [
    "model_label", "split",
    "roc_auc_baseline_with_grade_subgrade", "roc_auc_baseline_no_grade_subgrade", "roc_auc_delta",
    "pr_auc_baseline_with_grade_subgrade", "pr_auc_baseline_no_grade_subgrade", "pr_auc_delta",
    "f1_baseline_with_grade_subgrade", "f1_baseline_no_grade_subgrade", "f1_delta",
]
compact = ablation_deltas[compact_cols].copy()
compact = compact.rename(columns={"model_label": "model"})
save_table(compact, "grade_subgrade_ablation_compact")
display(compact)

if NO_GRADE_EXPORT_SUMMARY_PATH.exists():
    no_grade_export_summary = pd.read_csv(NO_GRADE_EXPORT_SUMMARY_PATH)
    save_table(no_grade_export_summary, "grade_subgrade_ablation_feature_summary")
    display(no_grade_export_summary)

for metric in ["f1", "pr_auc", "roc_auc", "precision", "recall"]:
    plot_df = ablation_metrics[ablation_metrics["split"] == "test"].copy()
    fig, ax = plt.subplots(figsize=(9, 4.5))
    labels = [MODEL_LABELS[m] for m in MODEL_FAMILIES]
    x = np.arange(len(labels))
    width = 0.35
    with_grade = plot_df[plot_df["feature_set"] == "baseline_with_grade_subgrade"].set_index("model_family").loc[MODEL_FAMILIES, metric]
    no_grade = plot_df[plot_df["feature_set"] == "baseline_no_grade_subgrade"].set_index("model_family").loc[MODEL_FAMILIES, metric]
    ax.bar(x - width/2, with_grade, width, label="with grade/subgrade")
    ax.bar(x + width/2, no_grade, width, label="without grade/subgrade")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=20, ha="right")
    ax.set_title(f"Grade/subgrade ablation on test: {metric}")
    ax.grid(axis="y", alpha=0.25)
    ax.legend()
    save_plot(fig, f"grade_subgrade_ablation_test_{metric}")


Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/tables/final_model_grade_subgrade_ablation_metrics.csv


,model_family,model_label,model,split,feature_set,roc_auc,pr_auc,brier_score,precision,recall,f1,threshold
9,hist_gradient_boosting,HistGradientBoosting,hist_gradient_boosting_no_grade_subgrade,test,baseline_no_grade_subgrade,0.709575,0.380035,0.151336,0.321367,0.688212,0.438140,0.182890
5,hist_gradient_boosting,HistGradientBoosting,hist_gradient_boosting_04,test,baseline_with_grade_subgrade,0.708493,0.377185,0.212358,0.319695,0.693143,0.437571,0.472937
7,logistic_regression,Logistic Regression,logistic_regression_no_grade_subgrade,test,baseline_no_grade_subgrade,0.697573,0.355817,0.155394,0.300132,0.750662,0.428814,0.164047
1,logistic_regression,Logistic Regression,logistic_regression_07,test,baseline_with_grade_subgrade,0.700061,0.361509,0.225017,0.310541,0.709320,0.431966,0.471801
11,random_forest,Random Forest,random_forest_no_grade_subgrade,test,baseline_no_grade_subgrade,0.704153,0.370329,0.203629,0.312537,0.714130,0.434789,0.443294
3,random_forest,Random Forest,random_forest_03,test,baseline_with_grade_subgrade,0.703753,0.371073,0.200271,0.321114,0.673832,0.434952,0.455266
8,hist_gradient_boosting,HistGradientBoosting,hist_gradient_boosting_no_grade_subgrade,validation,baseline_no_grade_subgrade,0.701741,0.425647,0.170437,0.365229,0.675751,0.474176,0.182890
4,hist_gradient_boosting,HistGradientBoosting,hist_gradient_boosting_04,validation,baseline_with_grade_subgrade,0.700829,0.423500,0.217717,0.359479,0.696715,0.474259,0.472937
6,logistic_regression,Logistic Regression,logistic_regression_no_grade_subgrade,validation,baseline_no_grade_subgrade,0.692454,0.408532,0.172715,0.344783,0.732726,0.468918,0.164047
0,logistic_regression,Logistic Regression,logistic_regression_07,validation,baseline_with_grade_subgrade,0.695407,0.412336,0.224051,0.355082,0.697669,0.470633,0.471801


Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/tables/final_model_grade_subgrade_ablation_deltas.csv


,model_family,model_label,split,f1_baseline_no_grade_subgrade,f1_baseline_with_grade_subgrade,pr_auc_baseline_no_grade_subgrade,pr_auc_baseline_with_grade_subgrade,precision_baseline_no_grade_subgrade,precision_baseline_with_grade_subgrade,recall_baseline_no_grade_subgrade,recall_baseline_with_grade_subgrade,roc_auc_baseline_no_grade_subgrade,roc_auc_baseline_with_grade_subgrade,roc_auc_delta,pr_auc_delta,f1_delta,precision_delta,recall_delta
0,hist_gradient_boosting,HistGradientBoosting,test,0.438140,0.437571,0.380035,0.377185,0.321367,0.319695,0.688212,0.693143,0.709575,0.708493,0.001082,0.002850,0.000569,0.001672,-0.004931
2,logistic_regression,Logistic Regression,test,0.428814,0.431966,0.355817,0.361509,0.300132,0.310541,0.750662,0.709320,0.697573,0.700061,-0.002488,-0.005692,-0.003152,-0.010409,0.041342
4,random_forest,Random Forest,test,0.434789,0.434952,0.370329,0.371073,0.312537,0.321114,0.714130,0.673832,0.704153,0.703753,0.000400,-0.000744,-0.000163,-0.008577,0.040298
1,hist_gradient_boosting,HistGradientBoosting,validation,0.474176,0.474259,0.425647,0.423500,0.365229,0.359479,0.675751,0.696715,0.701741,0.700829,0.000912,0.002147,-0.000083,0.005750,-0.020964
3,logistic_regression,Logistic Regression,validation,0.468918,0.470633,0.408532,0.412336,0.344783,0.355082,0.732726,0.697669,0.692454,0.695407,-0.002953,-0.003804,-0.001715,-0.010299,0.035057
5,random_forest,Random Forest,validation,0.471600,0.468373,0.416275,0.415081,0.351879,0.359521,0.714797,0.671762,0.696983,0.694572,0.002411,0.001194,0.003227,-0.007642,0.043035


Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/tables/final_model_grade_subgrade_ablation_compact.csv


...

,split,source_artifact,export_artifact,rows,original_columns,dropped_columns,export_columns
0,train,baseline_train_X.parquet,baseline_no_grade_subgrade_train_X.parquet,962641,100,42,58
1,validation,baseline_validation_X.parquet,baseline_no_grade_subgrade_validation_X.parquet,186920,100,42,58
2,test,baseline_test_X.parquet,baseline_no_grade_subgrade_test_X.parquet,195749,100,42,58


Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/plots/final_model_grade_subgrade_ablation_test_f1.png


Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/plots/final_model_grade_subgrade_ablation_test_pr_auc.png


Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/plots/final_model_grade_subgrade_ablation_test_roc_auc.png
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/plots/final_model_grade_subgrade_ablation_test_precision.png


Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/plots/final_model_grade_subgrade_ablation_test_recall.png


## 6. Recommendation

Accept `hist_gradient_boosting_04` as the final model and show the predicted-label volume for labels `0` and `1` on validation and test.

In [6]:

preferred_model_family = "hist_gradient_boosting"
preferred_candidate = "hist_gradient_boosting_04"
preferred_feature_set = "baseline_with_grade_subgrade"
preferred_operating_point = "best_validation_f1"

preferred_validation = selected_metrics[
    (selected_metrics["model_family"] == preferred_model_family) &
    (selected_metrics["model"] == preferred_candidate) &
    (selected_metrics["split"] == "validation") &
    (selected_metrics["operating_point"] == preferred_operating_point)
].iloc[0]
preferred_test = selected_metrics[
    (selected_metrics["model_family"] == preferred_model_family) &
    (selected_metrics["model"] == preferred_candidate) &
    (selected_metrics["split"] == "test") &
    (selected_metrics["operating_point"] == preferred_operating_point)
].iloc[0]

no_grade_test = ablation_metrics[
    (ablation_metrics["model_family"] == preferred_model_family) &
    (ablation_metrics["feature_set"] == "baseline_no_grade_subgrade") &
    (ablation_metrics["split"] == "test")
].iloc[0]

hgb_confusion_matrix = read_model_table(preferred_model_family, "confusion_matrix")
recommended_predicted_label_counts = (
    hgb_confusion_matrix[
        (hgb_confusion_matrix["candidate"] == preferred_candidate) &
        (hgb_confusion_matrix["split"].isin(["validation", "test"])) &
        (hgb_confusion_matrix["operating_point"] == preferred_operating_point)
    ]
    .groupby([
        "model_family", "candidate", "split", "operating_point", "threshold",
        "predicted_label", "predicted_class"
    ], as_index=False)["count"]
    .sum()
    .rename(columns={"count": "predicted_count"})
    .assign(split_order=lambda df: df["split"].map({"validation": 0, "test": 1}))
    .sort_values(["split_order", "predicted_label"])
    .drop(columns="split_order")
)
recommended_predicted_label_counts["predicted_share"] = (
    recommended_predicted_label_counts["predicted_count"] /
    recommended_predicted_label_counts.groupby("split")["predicted_count"].transform("sum")
).round(6)

def predicted_value(split: str, predicted_label: int, column: str) -> float:
    match = recommended_predicted_label_counts[
        (recommended_predicted_label_counts["split"] == split) &
        (recommended_predicted_label_counts["predicted_label"] == predicted_label)
    ]
    if match.empty:
        return np.nan
    value = match.iloc[0][column]
    return int(value) if column == "predicted_count" else float(value)

recommendation = pd.DataFrame([{
    "recommended_model_family": preferred_model_family,
    "recommended_model_label": MODEL_LABELS[preferred_model_family],
    "recommended_candidate": preferred_candidate,
    "recommended_feature_set": preferred_feature_set,
    "recommended_operating_point": preferred_operating_point,
    "selection_basis": (
        "Accept hist_gradient_boosting_04 as the final model because it is the selected "
        "HistGradientBoosting candidate and provides the strongest balanced F1/PR-AUC tradeoff "
        "for the final accepted-loan comparison. The no-grade/subgrade model remains documented "
        "as an ablation check, but is not the accepted final candidate."
    ),
    "validation_f1": preferred_validation["f1"],
    "validation_precision": preferred_validation["precision"],
    "validation_recall": preferred_validation["recall"],
    "validation_pr_auc": preferred_validation["pr_auc"],
    "validation_predicted_0_count": predicted_value("validation", 0, "predicted_count"),
    "validation_predicted_0_share": predicted_value("validation", 0, "predicted_share"),
    "validation_predicted_1_count": predicted_value("validation", 1, "predicted_count"),
    "validation_predicted_1_share": predicted_value("validation", 1, "predicted_share"),
    "test_f1": preferred_test["f1"],
    "test_precision": preferred_test["precision"],
    "test_recall": preferred_test["recall"],
    "test_pr_auc": preferred_test["pr_auc"],
    "test_predicted_0_count": predicted_value("test", 0, "predicted_count"),
    "test_predicted_0_share": predicted_value("test", 0, "predicted_share"),
    "test_predicted_1_count": predicted_value("test", 1, "predicted_count"),
    "test_predicted_1_share": predicted_value("test", 1, "predicted_share"),
    "test_f1_delta_vs_no_grade_subgrade": preferred_test["f1"] - no_grade_test["f1"],
    "test_pr_auc_delta_vs_no_grade_subgrade": preferred_test["pr_auc"] - no_grade_test["pr_auc"],
}])
save_table(recommendation, "recommendation")
display(recommendation)

save_table(recommended_predicted_label_counts, "predicted_label_counts")
display(recommended_predicted_label_counts)


Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/tables/final_model_recommendation.csv


,recommended_model_family,recommended_model_label,recommended_candidate,recommended_feature_set,recommended_operating_point,selection_basis,validation_f1,validation_precision,validation_recall,validation_pr_auc,...,test_f1,test_precision,test_recall,test_pr_auc,test_predicted_0_count,test_predicted_0_share,test_predicted_1_count,test_predicted_1_share,test_f1_delta_vs_no_grade_subgrade,test_pr_auc_delta_vs_no_grade_subgrade
0,hist_gradient_boosting,HistGradientBoosting,hist_gradient_boosting_04,baseline_with_grade_subgrade,best_validation_f1,Accept hist_gradient_boosting_04 as the final ...,0.474259,0.359479,0.696715,0.4235,...,0.437571,0.319695,0.693143,0.377185,106489,0.544008,89260,0.455992,-0.000569,-0.00285


Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/tables/final_model_predicted_label_counts.csv


,model_family,candidate,split,operating_point,threshold,predicted_label,predicted_class,predicted_count,predicted_share
2,hist_gradient_boosting,hist_gradient_boosting_04,validation,best_validation_f1,0.472937,0,Fully Paid,97524,0.521742
3,hist_gradient_boosting,hist_gradient_boosting_04,validation,best_validation_f1,0.472937,1,Charged Off,89396,0.478258
0,hist_gradient_boosting,hist_gradient_boosting_04,test,best_validation_f1,0.472937,0,Fully Paid,106489,0.544008
1,hist_gradient_boosting,hist_gradient_boosting_04,test,best_validation_f1,0.472937,1,Charged Off,89260,0.455992
